# Skill, Sequence, and Scoring — Paper Results

**Reproducing all results from:** *"Skill, Sequence, and Scoring: A Mathematical Comparison of Traditional and World Bowling Scoring Systems"* — Michael Borck, Curtin University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michael-borck/skill-sequence-scoring/blob/main/notebooks/01_paper_results.ipynb)

**Instructions:** Click *Runtime → Run all* to reproduce every figure and table from the paper. No installation required beyond standard Colab.

**Repository:** [github.com/michael-borck/skill-sequence-scoring](https://github.com/michael-borck/skill-sequence-scoring)

## Setup and Scoring Functions

All scoring functions are defined inline so this notebook runs standalone on Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict, Counter
from itertools import permutations
import statistics
import math

# ── Plot style (B&W-friendly) ────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.titlesize': 12,
    'axes.labelsize': 11, 'legend.fontsize': 9, 'figure.dpi': 150,
    'savefig.bbox': 'tight', 'axes.spines.top': False, 'axes.spines.right': False,
})
TC, WC = '#1a1a1a', '#888888'  # Traditional, World Bowling colours
TF, WF = '#2c2c2c', '#aaaaaa'  # Fill colours
TB = {'color': TF, 'edgecolor': TC, 'alpha': 0.85}
WB = {'color': WF, 'edgecolor': WC, 'hatch': '///', 'alpha': 0.85}
TL, WL = 'Traditional', 'World Bowling'

# ── Scoring functions ────────────────────────────────────────────────────────
def score_traditional(balls):
    score, i = 0, 0
    for frame in range(10):
        if i >= len(balls): return None
        if frame < 9:
            if balls[i] == 10:
                if i + 2 >= len(balls): return None
                score += 10 + balls[i+1] + balls[i+2]; i += 1
            else:
                if i + 1 >= len(balls): return None
                if balls[i] + balls[i+1] > 10: return None
                score += balls[i] + balls[i+1]
                if balls[i] + balls[i+1] == 10:
                    if i + 2 >= len(balls): return None
                    score += balls[i+2]
                i += 2
        else:
            if balls[i] == 10:
                if i + 2 >= len(balls): return None
                score += 10 + balls[i+1] + balls[i+2]
            else:
                if i + 1 >= len(balls): return None
                if balls[i] + balls[i+1] > 10: return None
                if balls[i] + balls[i+1] == 10:
                    if i + 2 >= len(balls): return None
                    score += 10 + balls[i+2]
                else:
                    score += balls[i] + balls[i+1]
    return score

def score_world(balls):
    score, i = 0, 0
    for frame in range(10):
        if i >= len(balls): return None
        if balls[i] == 10:
            score += 30; i += 1
        else:
            if i + 1 >= len(balls): return None
            if balls[i] + balls[i+1] > 10: return None
            if balls[i] + balls[i+1] == 10:
                score += 10 + balls[i]
            else:
                score += balls[i] + balls[i+1]
            i += 2
    return score

print("Scoring functions loaded ✓")

## Part 1: Exact Score Distributions (Section 3)

Computing exact score distributions using dynamic programming. This takes ~30 seconds.

In [ ]:
def traditional_distribution():
    dp = defaultdict(int)
    dp[(1, 1, 0, 0, 0, 0)] = 1
    while any(s[0] < 10 for s in dp):
        new_dp = defaultdict(int)
        for (frame, ball, first_ball, b1, b2, score), count in dp.items():
            if frame >= 10:
                new_dp[(frame, ball, first_ball, b1, b2, score)] += count; continue
            if ball == 1:
                for pins in range(11):
                    ns = score + pins * (1 + b1); nb1 = b2; nb2 = 0
                    if pins == 10: nb1 += 1; nb2 += 1; new_dp[(frame+1,1,0,nb1,nb2,ns)] += count
                    else: new_dp[(frame,2,pins,nb1,nb2,ns)] += count
            else:
                for pins in range(10 - first_ball + 1):
                    ns = score + pins * (1 + b1); nb1 = b2; nb2 = 0
                    if first_ball + pins == 10: nb1 += 1
                    new_dp[(frame+1,1,0,nb1,nb2,ns)] += count
        dp = new_dp
    results = defaultdict(int)
    for (frame, ball, first_ball, b1, b2, score), count in dp.items():
        for p1 in range(11):
            s1 = score + p1 * (1 + b1); nb1 = b2
            if p1 == 10:
                for p2 in range(11):
                    s2 = s1 + p2 * (1 + nb1)
                    if p2 == 10:
                        for p3 in range(11): results[s2 + p3] += count
                    else:
                        for p3 in range(10 - p2 + 1): results[s2 + p3] += count
            else:
                for p2 in range(10 - p1 + 1):
                    s2 = s1 + p2 * (1 + nb1)
                    if p1 + p2 == 10:
                        for p3 in range(11): results[s2 + p3] += count
                    else: results[s2] += count
    return dict(results)

def world_bowling_distribution():
    dp = defaultdict(int); dp[0] = 1
    for _ in range(10):
        new_dp = defaultdict(int)
        for score, count in dp.items():
            for first in range(11):
                if first == 10: new_dp[score + 30] += count
                else:
                    for second in range(10 - first + 1):
                        fs = (10 + first) if first + second == 10 else first + second
                        new_dp[score + fs] += count
        dp = new_dp
    return dict(dp)

print("Computing traditional distribution...")
trad_dist = traditional_distribution()
print(f"  Done — {len(trad_dist)} distinct scores")
print("Computing World Bowling distribution...")
world_dist = world_bowling_distribution()
print(f"  Done — {len(world_dist)} distinct scores")

### Table 1: Summary Statistics & Verification

In [ ]:
def analyse(dist, name):
    total = sum(dist.values())
    scores = sorted(dist.keys())
    mode = max(dist, key=dist.get)
    mean = sum(s * c for s, c in dist.items()) / total
    var = sum(c * (s - mean)**2 for s, c in dist.items()) / total
    cum = 0; median = None
    for s in scores:
        cum += dist[s]
        if cum * 2 >= total and median is None: median = s
    entropy = -sum((c/total) * math.log2(c/total) for c in dist.values() if c > 0)
    return {'name': name, 'total': total, 'mode': mode, 'mean': round(mean,1),
            'median': median, 'sd': round(var**0.5, 1), 'entropy': round(entropy, 4)}

ta, wa = analyse(trad_dist, TL), analyse(world_dist, WL)

print(f"{'Statistic':<26} {'Traditional':>18} {'World Bowling':>16}")
print("-" * 62)
for label, k in [('Total distinct games','total'), ('Mode','mode'), ('Mean','mean'),
                  ('Median','median'), ('Std deviation','sd'), ('Shannon entropy (bits)','entropy')]:
    print(f"{label:<26} {str(ta[k]):>18} {str(wa[k]):>16}")

# Verification against Balmoral Software
print("\n--- Verification ---")
checks = [(77, 172_542_309_343_731_946, 'trad mode'), (300, 1, 'perfect game'),
           (0, 1, 'all gutters')]
for s, expected, note in checks:
    actual = trad_dist.get(s, 0)
    print(f"  {'✓' if actual == expected else '✗'} Score {s}: {actual:,} ({note})")

impossible = [s for s in range(290, 300) if world_dist.get(s, 0) == 0]
print(f"\nImpossible World Bowling scores: {impossible}")

### Figure 1: Score Distribution Overlay & Figure 2: High-Score Tail

In [ ]:
scores = np.arange(301)
trad_counts = np.array([trad_dist.get(s, 0) for s in scores], dtype=float)
world_counts = np.array([world_dist.get(s, 0) for s in scores], dtype=float)
trad_p = trad_counts / trad_counts.sum()
world_p = world_counts / world_counts.sum()

# Figure 1: Full distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(scores, trad_p * 100, color=TC, linewidth=1.5, linestyle='-', label=TL)
ax.plot(scores, world_p * 100, color=WC, linewidth=1.5, linestyle='--', label=WL)
ax.set_xlabel('Score'); ax.set_ylabel('Probability (%)')
ax.set_title('Figure 1: Score Distributions'); ax.set_xlim(0, 300); ax.legend()
ax.axvline(77, color=TC, linestyle=':', alpha=0.4); ax.axvline(72, color=WC, linestyle=':', alpha=0.4)
ax.annotate('Mode=77', xy=(77, max(trad_p)*100), xytext=(85, max(trad_p)*100), fontsize=8, color=TC)
ax.annotate('Mode=72', xy=(72, max(world_p)*100), xytext=(40, max(world_p)*100*0.95), fontsize=8, color=WC)
plt.show()

# Figure 2: High-score tail
fig, ax = plt.subplots(figsize=(10, 5))
mask = scores >= 150
ax.semilogy(scores[mask], trad_p[mask], color=TC, linewidth=1.5, linestyle='-', label=TL)
ax.semilogy(scores[mask], world_p[mask], color=WC, linewidth=1.5, linestyle='--', label=WL)
for s in range(290, 300):
    if world_p[s] == 0: ax.axvline(s, color=WC, alpha=0.12, linewidth=4)
ax.annotate('290–299\nimpossible', xy=(294, 1e-20), fontsize=8, color=WC, ha='center', style='italic')
ax.set_xlabel('Score'); ax.set_ylabel('Probability (log scale)')
ax.set_title('Figure 2: High-Score Tail'); ax.set_xlim(150, 302); ax.legend()
plt.show()

## Sequence Sensitivity (Section 4)

### Figure 5: Small-multiples — 3 compositions

In [ ]:
X, sp, n10 = [10], [5,5], [5,4]

compositions = [
    ('2 Strikes + 7 Spares', {
        '2X then\n7 sp': X*2+sp*7+n10, '7 sp then\n2X': sp*7+X*2+n10,
        'X, 3sp,\nX, 4sp': X+sp*3+X+sp*4+n10}),
    ('5 Strikes + 4 Spares', {
        '5X then\n4 sp': X*5+sp*4+n10, '4 sp then\n5X': sp*4+X*5+n10,
        'Alt X/sp\n×4, X': (X+sp)*4+X+n10}),
    ('7 Strikes + 2 Spares', {
        '7X then\n2 sp': X*7+sp*2+n10, '2 sp then\n7X': sp*2+X*7+n10,
        'X,sp,\n5X,sp,X': X+sp+X*5+sp+X+n10}),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for idx, (title, seqs) in enumerate(compositions):
    ax = axes[idx]
    names = list(seqs.keys())
    ts = [score_traditional(b) for b in seqs.values()]
    ws = [score_world(b) for b in seqs.values()]
    x = np.arange(len(names)); w = 0.35
    ax.bar(x-w/2, ts, w, label=TL, **TB)
    ax.bar(x+w/2, ws, w, label=WL, **WB)
    ax.set_title(title); ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8)
    if idx == 0: ax.set_ylabel('Game Score'); ax.legend(fontsize=8)
    for i, (t, ww) in enumerate(zip(ts, ws)):
        ax.text(i-w/2, t+1, str(t), ha='center', fontsize=7, color=TC)
        ax.text(i+w/2, ww+1, str(ww), ha='center', fontsize=7, color=WC)
    ax.set_ylim(0, max(ts+ws)+25)
    ax.annotate(f'Trad. range: {max(ts)-min(ts)} pts', xy=(1, min(ts)-8),
                ha='center', fontsize=7, style='italic', color=TC)
fig.suptitle('Figure 5: Sequence Sensitivity — Same Frames, Different Order', fontsize=13, y=1.02)
fig.tight_layout(); plt.show()

## Reward Gradient (Section 5)

### Figure 3: Marginal value of consecutive strikes — 3 fill types

In [ ]:
fills = [('Weak open (3,2)', 3, 2), ('Mid open (5,4)', 5, 4), ('Strong open (8,1)', 8, 1)]
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for idx, (label, b1, b2) in enumerate(fills):
    ax = axes[idx]
    def game(n):
        balls = [10]*n + [b1, b2]*(10-n)
        if n == 10: balls = [10]*12
        elif n == 9: balls = [10]*9 + [10, b1, b2]
        return balls
    ts = [score_traditional(game(n)) for n in range(11)]
    ws = [score_world(game(n)) for n in range(11)]
    tm = [ts[i]-ts[i-1] for i in range(1, 11)]
    wm = [ws[i]-ws[i-1] for i in range(1, 11)]
    x = list(range(1, 11))
    ax.bar([i-0.18 for i in x], tm, 0.35, label=TL, **TB)
    ax.bar([i+0.18 for i in x], wm, 0.35, label=WL, **WB)
    ax.axhline(wm[0], color=WC, linestyle=':', alpha=0.5)
    ax.set_xlabel('Strike Number'); ax.set_title(label); ax.set_xticks(x)
    if idx == 0: ax.set_ylabel('Marginal Score Increase'); ax.legend(fontsize=8)

fig.suptitle('Figure 3: Marginal Value of Each Additional Consecutive Strike', fontsize=13, y=1.02)
fig.tight_layout(); plt.show()

## Part 2: Skill-Weighted Simulation (Sections 6–8)

### Simulation engine and skill tiers

In [ ]:
SKILL_TIERS = {
    'Recreational': {'p_strike': 0.04, 'p_spare': 0.27, 'pin_mean': 4.2},
    'Club':         {'p_strike': 0.20, 'p_spare': 0.47, 'pin_mean': 5.5},
    'Competitive':  {'p_strike': 0.40, 'p_spare': 0.64, 'pin_mean': 6.5},
    'Elite':        {'p_strike': 0.55, 'p_spare': 0.77, 'pin_mean': 7.2},
    'Professional': {'p_strike': 0.66, 'p_spare': 0.86, 'pin_mean': 7.8},
    'Top 10':       {'p_strike': 0.73, 'p_spare': 0.92, 'pin_mean': 8.2},
}

def sim_first_ball(rng, ps, pm):
    if rng.random() < ps: return 10
    p = min(pm / 10.0, 0.95); pins = rng.binomial(10, p)
    while pins == 10: pins = rng.binomial(10, p)
    return int(pins)

def sim_second_ball(rng, b1, psp):
    rem = 10 - b1
    if rem == 0: return 0
    if rng.random() < psp: return rem
    return int(rng.integers(0, rem))

def sim_game(rng, ps, psp, pm):
    balls = []
    for _ in range(9):
        b1 = sim_first_ball(rng, ps, pm); balls.append(b1)
        if b1 < 10: balls.append(sim_second_ball(rng, b1, psp))
    b1 = sim_first_ball(rng, ps, pm); balls.append(b1)
    if b1 == 10:
        b2 = sim_first_ball(rng, ps, pm); balls.append(b2)
        if b2 == 10: balls.append(sim_first_ball(rng, ps, pm))
        else: balls.append(sim_second_ball(rng, b2, psp))
    else:
        b2 = sim_second_ball(rng, b1, psp); balls.append(b2)
        if b1 + b2 == 10: balls.append(sim_first_ball(rng, ps, pm))
    return balls

def sim_game_momentum(rng, ps, psp, pm, mom=0.05):
    balls = []; p_eff = ps
    for _ in range(9):
        pc = max(0.01, min(0.99, p_eff))
        b1 = sim_first_ball(rng, pc, pm); balls.append(b1)
        if b1 == 10:
            p_eff = min(ps + 3*mom, ps + mom*(1 + (p_eff-ps)/max(mom,0.01)))
        else:
            b2 = sim_second_ball(rng, b1, psp); balls.append(b2)
            p_eff = ps if b1+b2 == 10 else ps - mom
    pc = max(0.01, min(0.99, p_eff))
    b1 = sim_first_ball(rng, pc, pm); balls.append(b1)
    if b1 == 10:
        pe = min(ps + 2*mom, 0.99)
        b2 = sim_first_ball(rng, pe, pm); balls.append(b2)
        if b2 == 10: balls.append(sim_first_ball(rng, pe, pm))
        else: balls.append(sim_second_ball(rng, b2, psp))
    else:
        b2 = sim_second_ball(rng, b1, psp); balls.append(b2)
        if b1+b2 == 10: balls.append(sim_first_ball(rng, ps, pm))
    return balls

def run_sim(n, ps, psp, pm, seed=42, momentum=False, mom=0.05):
    rng = np.random.default_rng(seed)
    ts, ws = [], []
    for _ in range(n):
        balls = sim_game_momentum(rng, ps, psp, pm, mom) if momentum else sim_game(rng, ps, psp, pm)
        t, w = score_traditional(balls), score_world(balls)
        if t is not None and w is not None: ts.append(t); ws.append(w)
    return np.array(ts), np.array(ws)

N_GAMES = 30_000
print(f"Simulation engine loaded ✓ ({N_GAMES:,} games per tier)")

### Figure 8: Score Distributions by Skill Tier (2×3 grid) & Figure 9: Mean Scores and Score Spread

In [ ]:
# Run simulations for all tiers
results = {}
for name, p in SKILL_TIERS.items():
    t, w = run_sim(N_GAMES, p['p_strike'], p['p_spare'], p['pin_mean'])
    results[name] = {'trad': t, 'world': w}
    print(f"  {name:15s}: trad mean={np.mean(t):.1f} (SD={np.std(t):.1f}), "
          f"world mean={np.mean(w):.1f} (SD={np.std(w):.1f})")

# Figure 8: 2x3 tier distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8)); axes = axes.flatten()
bins = np.arange(0, 305, 5)
for i, (name, p) in enumerate(SKILL_TIERS.items()):
    ax = axes[i]; r = results[name]
    ax.hist(r['trad'], bins=bins, alpha=0.7, color=TF, edgecolor=TC, linewidth=0.5, label=TL, density=True)
    ax.hist(r['world'], bins=bins, alpha=0.5, color=WF, edgecolor=WC, linewidth=0.5, hatch='///', label=WL, density=True)
    ax.set_title(f'{name}\n(strike rate: {p["p_strike"]*100:.0f}%)'); ax.set_xlabel('Score'); ax.set_ylabel('Density')
    if i == 0: ax.legend(fontsize=8)
    ax.axvline(np.mean(r['trad']), color=TC, linestyle='-', alpha=0.6, linewidth=1)
    ax.axvline(np.mean(r['world']), color=WC, linestyle='--', alpha=0.6, linewidth=1)
fig.suptitle('Figure 8: Score Distributions by Skill Tier', fontsize=14, y=1.02)
fig.tight_layout(); plt.show()

# Figure 9: Mean scores and score spread
tier_names = list(SKILL_TIERS.keys())
t_means = [np.mean(results[n]['trad']) for n in tier_names]
w_means = [np.mean(results[n]['world']) for n in tier_names]
t_sds = [np.std(results[n]['trad']) for n in tier_names]
w_sds = [np.std(results[n]['world']) for n in tier_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(tier_names))
ax1.bar(x-0.18, t_means, 0.35, label=TL, **TB)
ax1.bar(x+0.18, w_means, 0.35, label=WL, **WB)
ax1.set_xticks(x); ax1.set_xticklabels(tier_names, fontsize=9)
ax1.set_ylabel('Mean Score'); ax1.set_title('Mean Score by Skill Tier'); ax1.legend()
for i, (t, w) in enumerate(zip(t_means, w_means)):
    ax1.text(i-0.18, t+2, f'{t:.0f}', ha='center', fontsize=7, color=TC)
    ax1.text(i+0.18, w+2, f'{w:.0f}', ha='center', fontsize=7, color=WC)

ax2.plot(tier_names, t_sds, '-', marker='o', color=TC, label=TL, markersize=7)
ax2.plot(tier_names, w_sds, '--', marker='s', color=WC, label=WL, markersize=7)
ax2.set_ylabel('Score Standard Deviation'); ax2.set_title('Score Spread by Skill Tier')
ax2.legend(); ax2.tick_params(axis='x', rotation=15)
fig.suptitle('Figure 9', fontsize=13, y=1.02); fig.tight_layout(); plt.show()

### Figure 12: Momentum Model Comparison

Does adding streakiness (hot hand) amplify the traditional scoring advantage?

In [ ]:
tier_names = list(SKILL_TIERS.keys())
tsd_i, wsd_i, tsd_m, wsd_m = [], [], [], []

for name, p in SKILL_TIERS.items():
    ti, wi = run_sim(N_GAMES, p['p_strike'], p['p_spare'], p['pin_mean'])
    tm, wm = run_sim(N_GAMES, p['p_strike'], p['p_spare'], p['pin_mean'], momentum=True)
    tsd_i.append(np.std(ti)); wsd_i.append(np.std(wi))
    tsd_m.append(np.std(tm)); wsd_m.append(np.std(wm))
    print(f"  {name:15s}: indep gap={np.std(ti)-np.std(wi):+.1f}, momentum gap={np.std(tm)-np.std(wm):+.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(tier_names, tsd_i, '-', marker='o', color=TC, label='Trad. (independent)', markersize=7)
ax1.plot(tier_names, tsd_m, marker='^', color=TC, label='Trad. (momentum)', markersize=7, alpha=0.7, linestyle=':')
ax1.plot(tier_names, wsd_i, marker='s', color=WC, label='WB (independent)', markersize=7, linestyle='--')
ax1.plot(tier_names, wsd_m, marker='v', color=WC, label='WB (momentum)', markersize=7, alpha=0.7, linestyle=':')
ax1.set_ylabel('Score SD'); ax1.set_title('Score Spread: Independent vs Momentum'); ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=15)

x = np.arange(len(tier_names))
gap_i = [t-w for t,w in zip(tsd_i, wsd_i)]
gap_m = [t-w for t,w in zip(tsd_m, wsd_m)]
ax2.bar(x-0.18, gap_i, 0.35, label='Independent', **TB)
ax2.bar(x+0.18, gap_m, 0.35, label='Momentum', color='#555', edgecolor='#333', hatch='...', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(tier_names, fontsize=9, rotation=15)
ax2.set_ylabel('SD Gap (Trad − WB)'); ax2.set_title('Traditional Scoring Advantage'); ax2.legend(fontsize=8)
ax2.axhline(0, color='gray', linewidth=0.5)
fig.suptitle('Figure 12: Momentum Amplifies Traditional Scoring Advantage', fontsize=13, y=1.02)
fig.tight_layout(); plt.show()

print("\n✓ Momentum model confirms: independence assumption is conservative."
      "\n  The real-world effect is likely larger than our base simulation shows.")